In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
from __future__ import annotations

from typing import Iterable, Optional, Sequence, Union

class CSVDataset:
    """Wraps a pandas DataFrame loaded from CSV and adds EDA helpers.

    The original data is kept in ``self._original`` so you can always
    call ``reset()`` to undo any cleaning steps.
    """

    def __init__(
        self,
        filepath: Optional[str] = None,
        dataframe: Optional[pd.DataFrame] = None,
        **read_csv_kwargs,
    ):
        """
        Parameters
        ----------
        filepath : str, optional
            Path to a CSV file. Ignored if ``dataframe`` is given.
        dataframe : pd.DataFrame, optional
            Use an existing DataFrame instead of reading from disk.
        **read_csv_kwargs
            Passed straight through to ``pandas.read_csv``
            (e.g. sep=';', na_values=['NA', '?'], encoding='latin-1').
        """
        if dataframe is not None:
            self.df = dataframe.copy()
            self.filepath = None
        elif filepath is not None:
            self.filepath = filepath
            self.df = pd.read_csv(filepath, **read_csv_kwargs)
        else:
            raise ValueError("Provide either `filepath` or `dataframe`.")

        # Keep a pristine copy so cleaning is reversible.
        self._original = self.df.copy()

    # ------------------------------------------------------------------ #
    # Basic dunder / utility
    # ------------------------------------------------------------------ #
    def __repr__(self) -> str:
        rows, cols = self.df.shape
        return f"<CSVDataset rows={rows} cols={cols} source={self.filepath!r}>"

    def __len__(self) -> int:
        return len(self.df)

    def reset(self) -> "CSVDataset":
        """Restore the DataFrame to its state right after loading."""
        self.df = self._original.copy()
        return self

    def head(self, n: int = 5) -> pd.DataFrame:
        return self.df.head(n)

    # ------------------------------------------------------------------ #
    # Column type helpers
    # ------------------------------------------------------------------ #
    def numeric_columns(self) -> list[str]:
        """Columns that pandas treats as numeric."""
        return self.df.select_dtypes(include=[np.number]).columns.tolist()

    def categorical_columns(self, unique_threshold: Optional[int] = None) -> list[str]:
        """Columns that look categorical.

        By default this returns object / string / category dtype columns.
        If ``unique_threshold`` is given, numeric columns with fewer than
        that many distinct values are also flagged (e.g. a 0/1 flag column).
        """
        cat = self.df.select_dtypes(
            include=["object", "string", "category"]
        ).columns.tolist()

        if unique_threshold is not None:
            for col in self.numeric_columns():
                if self.df[col].nunique(dropna=True) < unique_threshold:
                    cat.append(col)
        return cat

    # ------------------------------------------------------------------ #
    # MISSING / NaN VALUE HANDLING
    # ------------------------------------------------------------------ #
    def missing_report(self) -> pd.DataFrame:
        """Per-column count and percentage of missing values, worst first."""
        count = self.df.isna().sum()
        pct = (count / len(self.df) * 100).round(2)
        report = (
            pd.DataFrame({"missing_count": count, "missing_pct": pct})
            .sort_values("missing_count", ascending=False)
        )
        return report[report["missing_count"] > 0]

    def has_missing(self) -> bool:
        return bool(self.df.isna().any().any())

    def drop_missing(
        self,
        axis: int = 0,
        how: str = "any",
        subset: Optional[Sequence[str]] = None,
        thresh: Optional[int] = None,
    ) -> "CSVDataset":
        """Drop rows (axis=0) or columns (axis=1) containing NaNs.

        Mirrors ``DataFrame.dropna``. Returns self for chaining.
        """
        self.df = self.df.dropna(axis=axis, how=how, subset=subset, thresh=thresh)
        return self

    def fill_missing(
        self,
        strategy: str = "mean",
        columns: Optional[Iterable[str]] = None,
        fill_value=None,
    ) -> "CSVDataset":
        """Impute missing values.

        Parameters
        ----------
        strategy : {'mean', 'median', 'mode', 'constant', 'ffill', 'bfill'}
            - mean / median : numeric columns only.
            - mode          : works on any column; uses the most frequent value.
            - constant      : fill with ``fill_value``.
            - ffill / bfill : forward / backward fill.
        columns : iterable of str, optional
            Limit imputation to these columns. Defaults to all columns.
        fill_value :
            Required when strategy == 'constant'.
        """
        cols = list(columns) if columns is not None else self.df.columns.tolist()

        if strategy == "constant":
            if fill_value is None:
                raise ValueError("strategy='constant' needs a `fill_value`.")
            self.df[cols] = self.df[cols].fillna(fill_value)

        elif strategy in ("ffill", "bfill"):
            method = strategy  # 'ffill' or 'bfill'
            self.df[cols] = getattr(self.df[cols], method)()

        elif strategy in ("mean", "median"):
            for col in cols:
                if pd.api.types.is_numeric_dtype(self.df[col]):
                    value = getattr(self.df[col], strategy)()
                    self.df[col] = self.df[col].fillna(value)
            # non-numeric columns are silently skipped for mean/median

        elif strategy == "mode":
            for col in cols:
                mode = self.df[col].mode(dropna=True)
                if not mode.empty:
                    self.df[col] = self.df[col].fillna(mode.iloc[0])

        else:
            raise ValueError(f"Unknown strategy: {strategy!r}")

        return self

    def replace_nan(
        self,
        value=None,
        numeric_strategy: str = "median",
        columns: Optional[Iterable[str]] = None,
        report: bool = False,
    ) -> "CSVDataset":
        """Replace all NaN / missing values in one call.
 
        This is the "just clean everything" shortcut. Unlike ``fill_missing``,
        which applies a single strategy, this handles numeric and string
        columns together in a single pass.
 
        Two modes:
 
        1. Blanket replace -- pass a ``value`` and every NaN across the
           selected columns becomes that value.
               ds.replace_nan(0)
               ds.replace_nan("unknown")
 
        2. Type-aware replace (default, when ``value`` is None) --
           numeric columns are filled with their mean/median (set by
           ``numeric_strategy``), and string/categorical columns are filled
           with their most frequent value (mode).
               ds.replace_nan()                          # median + mode
               ds.replace_nan(numeric_strategy="mean")   # mean + mode
 
        Parameters
        ----------
        value :
            If given, use this single value for every NaN (mode 1).
        numeric_strategy : {'mean', 'median'}
            How to fill numeric columns in type-aware mode.
        columns : iterable of str, optional
            Limit the replacement to these columns. Defaults to all.
        report : bool
            If True, print how many NaNs were replaced per column.
 
        Returns ``self`` so it can be chained.
        """
        cols = list(columns) if columns is not None else self.df.columns.tolist()
        before = self.df[cols].isna().sum()
 
        if value is not None:
            # Mode 1: blanket replacement with one fixed value.
            self.df[cols] = self.df[cols].fillna(value)
        else:
            # Mode 2: type-aware replacement.
            if numeric_strategy not in ("mean", "median"):
                raise ValueError("numeric_strategy must be 'mean' or 'median'.")
            for col in cols:
                if self.df[col].isna().sum() == 0:
                    continue
                if pd.api.types.is_numeric_dtype(self.df[col]):
                    fill = getattr(self.df[col], numeric_strategy)()
                else:
                    mode = self.df[col].mode(dropna=True)
                    fill = mode.iloc[0] if not mode.empty else value
                self.df[col] = self.df[col].fillna(fill)
 
        if report:
            replaced = before[before > 0]
            if replaced.empty:
                print("replace_nan: nothing to replace.")
            else:
                print("replace_nan: values replaced per column")
                for col, n in replaced.items():
                    print(f"  {col}: {int(n)}")
 
        return self

    # ------------------------------------------------------------------ #
    # CATEGORICAL VARIABLE HANDLING
    # ------------------------------------------------------------------ #
    def category_summary(self) -> dict[str, pd.Series]:
        """Value counts for each categorical column (handy for a quick look)."""
        return {
            col: self.df[col].value_counts(dropna=False)
            for col in self.categorical_columns()
        }

    def encode_categorical(
        self,
        method: str = "onehot",
        columns: Optional[Iterable[str]] = None,
        drop_first: bool = False,
    ) -> "CSVDataset":
        """Turn string/categorical columns into numbers.

        Parameters
        ----------
        method : {'onehot', 'label'}
            - onehot : one dummy column per category (pandas.get_dummies).
            - label  : map each category to an integer code.
        columns : iterable of str, optional
            Which columns to encode. Defaults to all detected categoricals.
        drop_first : bool
            For one-hot, drop the first level to avoid the dummy trap.

        Note: for 'label' encoding, the mapping learned per column is stored
        in ``self.label_maps`` so you can reverse it later.
        """
        cols = list(columns) if columns is not None else self.categorical_columns()

        if not cols:
            return self  # nothing to do

        if method == "onehot":
            self.df = pd.get_dummies(
                self.df, columns=cols, drop_first=drop_first
            )

        elif method == "label":
            if not hasattr(self, "label_maps"):
                self.label_maps: dict[str, dict] = {}
            for col in cols:
                codes, uniques = pd.factorize(self.df[col])
                # factorize marks NaN as -1; keep it as NaN instead
                codes = pd.Series(codes, index=self.df.index).replace(-1, np.nan)
                self.df[col] = codes
                self.label_maps[col] = dict(enumerate(uniques))

        else:
            raise ValueError(f"Unknown method: {method!r}")

        return self

    # ------------------------------------------------------------------ #
    # EDA CONVENIENCE
    # ------------------------------------------------------------------ #
    def summary(self) -> dict:
        """A one-glance overview dictionary."""
        return {
            "shape": self.df.shape,
            "columns": self.df.columns.tolist(),
            "dtypes": self.df.dtypes.astype(str).to_dict(),
            "numeric_columns": self.numeric_columns(),
            "categorical_columns": self.categorical_columns(),
            "total_missing": int(self.df.isna().sum().sum()),
            "duplicate_rows": int(self.df.duplicated().sum()),
            "memory_kb": round(self.df.memory_usage(deep=True).sum() / 1024, 1),
        }

    def describe(self, include_all: bool = True) -> pd.DataFrame:
        """Descriptive stats. ``include_all`` also covers categorical cols."""
        return self.df.describe(include="all" if include_all else None)

    def correlations(self, method: str = "pearson") -> pd.DataFrame:
        """Correlation matrix across numeric columns."""
        return self.df[self.numeric_columns()].corr(method=method)

    def outlier_flags(self, column: str, k: float = 1.5) -> pd.Series:
        """Boolean mask of IQR-based outliers in a numeric column.

        A value is an outlier if it falls outside
        [Q1 - k*IQR, Q3 + k*IQR]. Default k=1.5 is the usual Tukey rule.
        """
        if column not in self.numeric_columns():
            raise ValueError(f"{column!r} is not numeric.")
        q1, q3 = self.df[column].quantile([0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - k * iqr, q3 + k * iqr
        return (self.df[column] < low) | (self.df[column] > high)

In [3]:
#df_train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv',sep=',')
df_test = CSVDataset("/kaggle/input/competitions/titanic/test.csv", na_values=["", "NA", "?"])

In [4]:
df_test=df_test.replace_nan(numeric_strategy="mean")
df_test=df_test.encode_categorical(method="label")

In [5]:
df_train = CSVDataset("/kaggle/input/competitions/titanic/train.csv", na_values=["", "NA", "?"])

In [6]:
def section(title: str) -> None:
    """Print a small header so the output is easy to scan."""
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

section("OVERVIEW")
print(df_train)                       # <CSVDataset rows=... cols=...>
print(df_train.head(3))               # first few rows



OVERVIEW
<CSVDataset rows=891 cols=12 source='/kaggle/input/competitions/titanic/train.csv'>
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  


In [7]:
section("SUMMARY")
for key, value in df_train.summary().items():
    print(f"{key:>20}: {value}")

section("MISSING VALUE REPORT")
print(df_train.missing_report())

section("CATEGORICAL COLUMNS")
print("Detected:", df_train.categorical_columns())

for col, counts in df_train.category_summary().items():
        print(f"\n[{col}]")
        print(counts)

section("DESCRIPTIVE STATS")
print(df_train.describe())

section("CORRELATIONS (numeric)")
print(df_train.correlations().round(2))

# section("OUTLIERS IN 'age' (Tukey IQR)")
# flags = df_train.outlier_flags("age")
# print("Outlier rows:", df_train.df.loc[flags, "age"].tolist())

    # 2. Clean it up.
    #    - numeric gaps -> median
    #    - categorical gaps -> most frequent value
section("CLEANING")
numeric = df_train.numeric_columns()
categorical = df_train.categorical_columns()

df_train.fill_missing(strategy="median", columns=numeric)
df_train.fill_missing(strategy="mode", columns=categorical)
print(f"Missing values remaining: {df_train.df.isna().sum().sum()}")

    # 3. Encode categoricals two different ways, on fresh copies each time.
section("ONE-HOT ENCODING")
onehot = CSVDataset(dataframe=df_train.df)
onehot.encode_categorical(method="onehot", drop_first=True)
print(onehot.df.head())

section("LABEL ENCODING")
labeled = CSVDataset(dataframe=df_train.df)
labeled.encode_categorical(method="label")
print(labeled.df.head())
# print("\nLabel maps (code -> original value):")
# for col, mapping in labeled.label_maps.items():
#     print(f"  {col}: {mapping}")

# 4. reset() proves the original data is still intact.
section("RESET DEMO")
print("Before reset:", df_train.df["Survived"].isna().sum())
df_train.reset()
print("After reset :", df_train.df["Survived"].isna().sum())



SUMMARY
               shape: (891, 12)
             columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
              dtypes: {'PassengerId': 'int64', 'Survived': 'int64', 'Pclass': 'int64', 'Name': 'object', 'Sex': 'object', 'Age': 'float64', 'SibSp': 'int64', 'Parch': 'int64', 'Ticket': 'object', 'Fare': 'float64', 'Cabin': 'object', 'Embarked': 'object'}
     numeric_columns: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
 categorical_columns: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
       total_missing: 866
      duplicate_rows: 0
           memory_kb: 285.6

MISSING VALUE REPORT
          missing_count  missing_pct
Cabin               687        77.10
Age                 177        19.87
Embarked              2         0.22

CATEGORICAL COLUMNS
Detected: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']

[Name]
Name
Dooley, Mr. Patrick                                    1
Braund

In [8]:
print(dir(df_train))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_original', 'categorical_columns', 'category_summary', 'correlations', 'describe', 'df', 'drop_missing', 'encode_categorical', 'filepath', 'fill_missing', 'has_missing', 'head', 'missing_report', 'numeric_columns', 'outlier_flags', 'replace_nan', 'reset', 'summary']


In [9]:
df_train.fill_missing().df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000000,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000000,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000000,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000000,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.000000,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.000000,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.000000,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,29.699118,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.000000,0,0,111369,30.0000,C148,C


In [10]:
df_train=df_train.replace_nan(numeric_strategy="mean")
df_train=df_train.encode_categorical(method="label")


In [11]:
y_train=df_train.df['Survived']
x_train=df_train.df.drop(columns=['PassengerId','Survived'],axis=1)

In [12]:
x_test=df_test.df.drop('PassengerId',axis=1)

In [13]:
import xgboost as xgb

In [14]:
dtrain = xgb.DMatrix(x_train, label=y_train, enable_categorical=True)
dtest = xgb.DMatrix(x_test)

In [15]:
# 3. Define parameters and train using native API
params = {
    'objective': 'binary:logistic',
    'max_depth': 6,
    'learning_rate': 0.05
}

# Train the model
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=100,
)

# 4. Predict using the DMatrix
predictions = model.predict(dtrain)
y_pred = model.predict(dtest)

In [16]:
predictions=(predictions >= 0.5).astype(int)

In [17]:
from sklearn.metrics import confusion_matrix

In [18]:
import seaborn as sns

cm=confusion_matrix(y_train,predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,  # Shows the raw numbers inside the squares
    fmt="d",  # Formats numbers as integers (stops it from using scientific notation)
    cmap="Blues",  # Sets the color palette
    xticklabels=["0T", "1T"],  # Custom class labels for columns
    yticklabels=["0", "1"],  # Custom class labels for rows
)

# 4. Add labels and show plot
plt.ylabel("Actual Labels")
plt.xlabel("Predicted Labels")
plt.title("Confusion Matrix")
plt.show()

NameError: name 'plt' is not defined